#Spatial Heterogeneity Analysis Among Distinct Cellular Subpopulations Within A Single Heterogenous Tumour Organoid


In [3]:
#Importing Required Libraries And Setting Up Visualization Style
import numpy as np, pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,silhouette_samples,adjusted_rand_score

from sklearn.model_selection import train_test_split as ttsplit,cross_val_score,StratifiedKFold
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

try:
    import umap
    umap_avail=True
except ImportError:
    umap_avail=False
    print("not Available")

try:
    import shap
    shap_avail=True
except ImportError:
    shap_avail=False
    print("Not Available")

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)
plt.rcParams["font.size"]=10

In [ ]:
#Loading Data
def load_data(data_path):
    fpath=f"{data_path}/41592_2025_2685_MOESM12_ESM.xlsx"
    df=pd.read_excel(fpath)
    print(f"Dataset Loaded: {df.shape[0]} Rows × {df.shape[1]} Columns")

    meta_cols=[i for i in df.columns if not i.startswith('Value of')]
    feat_cols=[i for i in df.columns if i.startswith('Value of')]
    print(f"\nMetadata Columns ({len(meta_cols)}):")
    for i in meta_cols:
        print(f">{i}")
    print(f"\nFeature Columns ({len(feat_cols)}):")

    #Grouping Features By Type For Display
    intensity_feat=[i for i in feat_cols if 'intensity' in i.lower() or 'cv' in i.lower() or 'mean' in i.lower()]
    topology_feat=[i for i in feat_cols if 'neighbor' in i.lower() or 'distance' in i.lower() or 'density' in i.lower()]
    shape_feat=[i for i in feat_cols if 'prolate' in i.lower() or 'oblate' in i.lower() or 'axis' in i.lower() or 'ratio' in i.lower()]

    if intensity_feat:
        print(f"\nIntensity Features ({len(intensity_feat)}):")
        for i in intensity_feat[:5]:
            print(f"• {i.replace('Value of Default_','')[:50]}")
        if len(intensity_feat)>5:
            print(f"... And {len(intensity_feat)-5} More")

    if topology_feat:
        print(f"\nTopology Features ({len(topology_feat)}):")
        for i in topology_feat[:5]:
            print(f"• {i.replace('Value of Default','')[:50]}")
        if len(topology_feat)>5:
            print(f"... And {len(topology_feat)-5} More")

    if shape_feat:
        print(f"\nShape Features: ({len(shape_feat)}):")
        for i in shape_feat:
            print(f"• {i.replace('Value of Default','')[:50]}")

    print(f"\nThis Is A Single Large Organoid With Spatial Heterogeneity")
    print(f"Total Cells Analyzed: {len(df)}")

    #Checking If We've Position Data
    pos_cols=[i for i in meta_cols if 'position' in i.lower() or '_x' in i.lower() or '_y' in i.lower() or '_z' in i.lower()]
    if pos_cols:
        print(f"\nSpatial Position Data Available: {pos_cols}")
    else:
        print(f"\nNo Explicit Position Columns Found-> Will Need To Infer From Features")
    return df,feat_cols,meta_cols

data_path="/content/drive/MyDrive/Organoid Analysis"
df_fig5b,feat_cols,meta_cols=load_data(data_path)

In [ ]:
#Exploring The Dataset
def explore(df,feat_cols):
    print(f"Total Cells: {len(df)}")
    print(f"Total Features: {len(feat_cols)}")

    #Checking For Missing Values
    miss_counts=df[feat_cols].isnull().sum()
    if miss_counts.sum()>0:
        print(f"\nFeatures With Missing Values:")
        for i,j in miss_counts[miss_counts>0].items():
            feat=i.replace('Value of Default','')[:40]
            print(f"{feat}: {j} Missing: ({j/len(df)*100:.1f}%)")
    else:
        print(f"\nNo Missing Values In Feature Columns")

    #Selecting Representative Features If We've Prolate/Oblate
    keyfeat=[]
    for col in feat_cols:
        if 'prolate' in col.lower() or 'oblate' in col.lower():
            keyfeat.append(col)
    if not keyfeat:
        # If no prolate/oblate, show first 5 features
        keyfeat=feat_cols[:5]
    if keyfeat:
        stats_df=df[keyfeat].describe()
        print(stats_df.to_string())

explore(df_fig5b,feat_cols)

In [ ]:
#Prolate/Oblate Analysis
def analyze(df,feat_cols):
    print("These Ratios Describe 3D Shape Anisotropy:")
    print("• Prolate: (Major-Medium)/(Major-Minor)")
    print("→ High Values=Rod-Like (Elongated)")
    print("• Oblate: (Medium-Minor)/(Major-Minor)")
    print("→ High Values=Disk-Like (Flattened)")
    print("\nShape Categories:")
    print("• Rod: High Prolate, Low oblate")
    print("• Disk: Low Prolate, High Oblate")
    print("• Sphere: Both Moderate (~0.5)")

    #Finding Prolate And Oblate Columns
    pro_col=None
    ob_col=None
    for i in feat_cols:
        if 'prolate' in i.lower():
            pro_col=i
        if 'oblate' in i.lower():
            ob_col=i
    if pro_col is None or ob_col is None:
        print("\nProlate/Oblate Columns Not Found In Standard Format")
        print("Looking For Shape Ratio Columns...")
        #Trying To Find Alternative Shape Columns
        shape_cols=[i for i in feat_cols if 'ratio' in i.lower()]
        if len(shape_cols)>=2:
            pro_col=shape_cols[0]
            ob_col=shape_cols[1]
            print(f"Using: {pro_col}")
            print(f"Using: {ob_col}")
        else:
            print("Couldn't Identify Shape Ratio Columns")
            return df,None,None

    print("\nProlate/Oblate Distribution\n")
    pro_val=df[pro_col].dropna()
    ob_val=df[ob_col].dropna()
    print(f"Prolate Ratio:")
    print(f"Mean: {pro_val.mean():.3f}")
    print(f"Median: {pro_val.median():.3f}")
    print(f"Range: [{pro_val.min():.3f}, {pro_val.max():.3f}]")
    print(f"\nOblate Ratio:")
    print(f"Mean: {ob_val.mean():.3f}")
    print(f"Median: {ob_val.median():.3f}")
    print(f"Range: [{ob_val.min():.3f}, {ob_val.max():.3f}]")

    #Following Typical Shape Analysis Conventions
    df_copy=df.copy()
    #Defining Thresholds Based On Actual Data Distribution (25th/75th Percentiles)
    pro_high=np.percentile(pro_val,75)  #~0.35
    pro_low=np.percentile(pro_val,25)   #~0.11
    ob_high=np.percentile(ob_val,75)    #~0.28
    ob_low=np.percentile(ob_val,25)     #~0.03

    print(f"\nData-Driven Thresholds (25th/75th Percentiles):")
    print(f"Prolate:\n Low (25th Percentile)={pro_low:.3f}")
    print(f"High (75th Percentile)={pro_high:.3f}")
    print(f"Oblate:\n Low (25th Percentile)={ob_low:.3f}")
    print(f"High (75th Percentile)={ob_high:.3f}")
    print(f"\nClassification Rules:")
    print(f"• Rod: Prolate>{pro_high:.3f} And Oblate<{ob_low:.3f}")
    print(f"• Disk: Prolate<{pro_low:.3f} And Oblate>{ob_high:.3f}")
    print(f"• Sphere: Everything Else (Intermediate Values)")

    def categorize_shape(row):
        p=row[pro_col]
        o=row[ob_col]
        if pd.isna(p) or pd.isna(o):
            return 'Unknown'
        if p>pro_high and o<ob_low:
            return 'Rod'
        elif p<pro_low and o>ob_high:
            return 'Disk'
        else:
            return 'Sphere'
    df_copy['Shape_Category']=df_copy.apply(categorize_shape,axis=1)
    print("\nShape Category Distribution\n")

    shape_cnt=df_copy['Shape_Category'].value_counts()
    for i,j in shape_cnt.items():
        print(f"{i}: {j} cells ({j/len(df_copy)*100:.1f}%)")

    #Visualizing Prolate Vs Oblate Scatter
    print("\nGenerating Prolate Vs. Oblate Scatter Plot...\n")
    fig,ax=plt.subplots(figsize=(10,8))

    #Coloring By Shape Category
    for i in ['Rod','Disk','Sphere']:
        mask=df_copy['Shape_Category']==i
        if mask.sum()>0:
            ax.scatter(df_copy.loc[mask,pro_col],
                      df_copy.loc[mask,ob_col],
                      label=i,alpha=0.5,s=20)
    ax.set_xlabel('Prolate Ratio',fontsize=12)
    ax.set_ylabel('Oblate Ratio',fontsize=12)
    ax.set_title('Prolate Vs. Oblate Ratio: Shape Distribution',fontsize=14)
    ax.legend()
    ax.grid(True,alpha=0.3)

    #Adding Reference Lines
    ax.axvline(0.5,color='gray',linestyle='--',alpha=0.5,linewidth=1)
    ax.axhline(0.5,color='gray',linestyle='--',alpha=0.5,linewidth=1)
    plt.tight_layout()
    plt.show()
    return df_copy,pro_col,ob_col

df_fig5b,pro_col,ob_col=analyze(df_fig5b,feat_cols)

In [ ]:
#Discovering Regions Using Unsupervised Machine Learning
def optimal_clusters(x_scaled,max_k=10):
    print("Testing k From 2 To {max_k}...\n")
    inertias=[]
    silhouette_scr=[]
    k_val=range(2,max_k+1)
    for k in k_val:
        #Fitting K-Means
        kmeans=KMeans(n_clusters=k,random_state=42,n_init=10)
        cluster_lbl=kmeans.fit_predict(x_scaled)
        #Calculating Metrics
        inertias.append(kmeans.inertia_)
        sil_score=silhouette_score(x_scaled,cluster_lbl)
        silhouette_scr.append(sil_score)
        print(f"k={k}: Inertia={kmeans.inertia_:.0f}, Silhouette={sil_score:.3f}")

    #Finding Optimal k (Highest Silhouette Score)
    opt_k=k_val[np.argmax(silhouette_scr)]
    print(f"\nRecommended k: {opt_k} (Highest Silhouette Score: {max(silhouette_scr):.3f})\n")
    #Plotting Metrics
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    #Elbow Plot
    axes[0].plot(k_val,inertias,'bo-',linewidth=2,markersize=8)
    axes[0].set_xlabel('Number Of Clusters (k)',fontsize=12)
    axes[0].set_ylabel('Within-Cluster Sum Of Squares',fontsize=12)
    axes[0].set_title('Elbow Method',fontsize=13)
    axes[0].grid(True,alpha=0.3)
    axes[0].axvline(opt_k,color='red',linestyle='--',alpha=0.7,label=f'Optimal k={opt_k}')
    axes[0].legend()
    #Silhouette Plot
    axes[1].plot(k_val,silhouette_scr,'go-',linewidth=2,markersize=8)
    axes[1].set_xlabel('Number Of Clusters (k)',fontsize=12)
    axes[1].set_ylabel('Silhouette Score',fontsize=12)
    axes[1].set_title('Silhouette Analysis',fontsize=13)
    axes[1].grid(True,alpha=0.3)
    axes[1].axvline(opt_k,color='red',linestyle='--',alpha=0.7,label=f'Optimal k={opt_k}')
    axes[1].axhline(0.5,color='gray',linestyle=':',alpha=0.5,label='Good Threshold (0.5)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    return opt_k,inertias,silhouette_scr


def perform_kmeans(df,feat_cols,n_clusters=3):
    #Preparing Data
    x=df[feat_cols].values
    #Handling Any Missing Values
    if np.isnan(x).any():
        print("\nMissing Values Detected,Imputing With Feature Means...")
        from sklearn.impute import SimpleImputer
        imputer=SimpleImputer(strategy='mean')
        x=imputer.fit_transform(X)

    #Standardizing Features
    std=StandardScaler()
    x_scaled=std.fit_transform(x)
    print(f"Scaled Feature Matrix: {x_scaled.shape}")
    print(f"Mean: {x_scaled.mean():.3f}, Std: {x_scaled.std():.3f}")
    #Fitting K-Means
    print(f"\nRunning K-means With {n_clusters} Clusters...")
    kmeans=KMeans(n_clusters=n_clusters,random_state=42,n_init=10)
    cluster_lbl=kmeans.fit_predict(x_scaled)

    #Addding Cluster Labels To Dataframe
    df_clustered=df.copy()
    df_clustered['Cluster']=cluster_lbl

    #Calculating Silhouette Score
    sil_score=silhouette_score(x_scaled,cluster_lbl)
    print(f"Silhouette Score: {sil_score:.3f}")
    if sil_score>0.5:
        print("Interpretation: Good Cluster Separation")
    elif sil_score>0.3:
        print("Interpretation: Moderate Cluster Separation")
    else:
        print("Interpretation: Weak Cluster Separation")
    print(f"\nCluster Sizes:")
    cluster_cnt=df_clustered['Cluster'].value_counts().sort_index()
    for i,j in cluster_cnt.items():
        print(f"Cluster {i}: {j} Cells ({j/len(df_clustered)*100:.1f}%)")
    return df_clustered,x_scaled,std,kmeans


def visualize_clusters(df_clustered,x_scaled,n_clusters):
    #PCA
    pca=PCA(n_components=2,random_state=42)
    x_pca=pca.fit_transform(x_scaled)
    print(f"Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")
    print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
    print(f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

    #UMAP (If Available)
    if umap_avail:
        print("\nRunning UMAP (2 Components)...")
        reducer=umap.UMAP(n_components=2,random_state=42,n_neighbors=15,min_dist=0.1)
        x_umap=reducer.fit_transform(x_scaled)
    else:
        x_umap=None
        print("\nUMAP Not Available,Skipping UMAP Visualization")
    print("\nGenerating Visualization...\n")
    if x_umap is not None:
        fig,axes=plt.subplots(1,2,figsize=(16,7))
    else:
        fig,axes=plt.subplots(1,1,figsize=(10,8))
        axes=[axes]
    #PCA Plot
    ax=axes[0]
    scatter=ax.scatter(x_pca[:,0],x_pca[:,1],
                        c=df_clustered['Cluster'],cmap='tab10',
                        alpha=0.6,s=20)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',fontsize=12)
    ax.set_title('K-Means Clusters: PCA Projection',fontsize=14)
    plt.colorbar(scatter,ax=ax,label='Cluster')
    ax.grid(True,alpha=0.3)

    #UMAP Plot
    if x_umap is not None:
        ax=axes[1]
        scatter=ax.scatter(x_umap[:,0],x_umap[:,1],
                            c=df_clustered['Cluster'],cmap='tab10',
                            alpha=0.6,s=20)
        ax.set_xlabel('UMAP 1',fontsize=12)
        ax.set_ylabel('UMAP 2',fontsize=12)
        ax.set_title('K-Means Clusters: UMAP Projection',fontsize=14)
        plt.colorbar(scatter,ax=ax,label='Cluster')
        ax.grid(True,alpha=0.3)
    plt.tight_layout()
    plt.show()
    return x_pca,x_umap,pca

x_scaled_data=StandardScaler().fit_transform(
    df_fig5b[feat_cols].fillna(df_fig5b[feat_cols].mean())
)
opt_k,inertia_val,silhouette_val=optimal_clusters(x_scaled_data,max_k=8)
df_fig5b_clustered,x_scaled_fig5b,scaler_fig5b,kmeans_mdl=perform_kmeans(
    df_fig5b,feat_cols,n_clusters=opt_k
)
x_pca_fig5b,x_umap_fig5b,pca_mdl=visualize_clusters(
    df_fig5b_clustered,x_scaled_fig5b,opt_k
)



In [ ]:
#Validating Our Findings Against Biology
def validate(df,pro_col,ob_col):
    if 'Shape_Category' not in df.columns or 'Cluster' not in df.columns:
        print("\nMissing Required Columns For Validation")
        return None
    print("\nComparing Unsupervised Custers With Shape Categories\n:")

    #Filtering Out Unknown Shapes
    df_valid=df[df['Shape_Category']!='Unknown'].copy()
    #Creating Crosstab
    crosstab=pd.crosstab(df_valid['Cluster'],df_valid['Shape_Category'],
                           margins=True,margins_name='Total')
    print("Cluster Vs. Shape Category:")
    print(crosstab)

    #Converting Shape Categories To Numeric For ARI Calculation
    shape_numeric=pd.Categorical(df_valid['Shape_Category']).codes
    cluster_numeric=df_valid['Cluster'].values

    ari_score=adjusted_rand_score(shape_numeric,cluster_numeric)
    print(f"Adjusted Rand Index: {ari_score:.3f}")
    if ari_score>0.5:
        print("Interpretation: Strong Agreement-> Clusters Align Well With Shapes")
    elif ari_score>0.2:
        print("Interpretation: Moderate Agreement-> Some Alignment Between Clusters And Shapes")
    else:
        print("Interpretation: Weak Agreement-> Clusters Capture Different Patterns Than Shapes")
    print("\nInterpretation:")
    print("• ARI=1: Perfect Agreement")
    print("• ARI=0: Random Agreement")
    print("• ARI<0: Less Agreement Than Random")

    #Visualizing Agreement
    print("\nGenerating Cluster-Shape Agreement Visualization...\n")
    #Heatmap Of Crosstab (Without Margins)
    crosstab_norm=crosstab.iloc[:-1,:-1]
    fig,ax=plt.subplots(figsize=(10,8))
    sns.heatmap(crosstab_norm,annot=True,fmt='d',cmap='YlOrRd',ax=ax,
               cbar_kws={'label':'Cell Count'})
    return ari_score

if pro_col and ob_col:
    ari_valid=validate(df_fig5b_clustered,pro_col,ob_col)


In [ ]:
#Characterizing Clusters
def characterize(df,feat_cols,n_clusters):
    print("\nComparing Clusters Across All Features:\n")
    #Preparing Clean Data
    x=df[feat_cols].fillna(df[feat_cols].mean()).values
    clusters=df['Cluster'].values
    #Storing Results
    feat_stats=[]
    print("\nComputing Statistics For Each Feature...\n")
    for i,j in enumerate(feat_cols):
        feat_data=x[:,i]
        #Getting Values Per Cluster
        cluster_means=[]
        for k in range(n_clusters):
            cluster_mask=clusters==k
            cluster_means.append(feat_data[cluster_mask].mean())

        #ANOVA Test (Are Means Significantly Different Across Clusters?)
        grp=[feat_data[clusters==i] for i in range(n_clusters)]
        f_stat,p_val=stats.f_oneway(*grp)

        #Effect Size (Eta-Squared)
        ss_btwn=sum([len(i)*(np.mean(i)-np.mean(feat_data))**2 for i in grp])
        ss_total=np.sum((feat_data-np.mean(feat_data))**2)
        eta_sq=ss_btwn/ss_total if ss_total>0 else 0
        feat_stats.append({
            'Feature':j.replace('Value of Default','')[:40],
            'F_statistic':f_stat,
            'p_value':p_val,
            'eta_squared':eta_sq,
            'Cluster_0_mean':cluster_means[0],
            'Cluster_1_mean':cluster_means[1] if n_clusters>1 else np.nan,
            'Cluster_2_mean':cluster_means[2] if n_clusters>2 else np.nan,
        })
    #Creating DataFrame
    comp_df=pd.DataFrame(feat_stats)
    comp_df=comp_df.sort_values('eta_squared',ascending=False)

    #Identifying Top Discriminative Features
    top_feat=comp_df[comp_df['p_value']<0.05].head(15)
    print("\nTop 15 Features That Seperate Clusters (p<0.05)\n")
    print(f"{'Feature':<42} {'η²':<8} {'p-value':<10}")
    for _, row in top_feat.iterrows():
        print(f"{row['Feature']:<42} {row['eta_squared']:<8.3f} {row['p_value']:<10.2e}")

    #Visualizing Top 6 Features
    print("\nGenerating Feature Comparison Visualizations...\n")
    top6_feat=top_feat.head(6)
    fig,axes=plt.subplots(2,3,figsize=(16,10))
    axes=axes.flatten()

    for idx,(_, row) in enumerate(top6_feat.iterrows()):
        ax=axes[idx]
        #Getting Original Feature Name
        feat_org=[i for i in feat_cols if row['Feature'] in i][0]
        feat_data=df[feat_org].fillna(df[feat_org].mean())
        #Violin Plot
        to_plot=[feat_data[df['Cluster']==i].values for i in range(n_clusters)]
        parts=ax.violinplot(to_plot,positions=range(n_clusters),
                             showmeans=True,showmedians=True)
        ax.set_xlabel('Cluster',fontsize=10)
        ax.set_ylabel('Value',fontsize=10)
        ax.set_title(f"{row['Feature']}\n(η²={row['eta_squared']:.3f}, p={row['p_value']:.2e})",
                    fontsize=10)
        ax.set_xticks(range(n_clusters))
        ax.grid(True,alpha=0.3,axis='y')
    plt.tight_layout()
    plt.show()
    return comp_df,top_feat['Feature'].tolist()

feat_comp,top_discriminative=characterize(
    df_fig5b_clustered,
    feat_cols,
    opt_k
)



In [ ]:
#Feature Importance For Clustering
def feat_importance(df,feat_cols,n_clusters):
    print("\nUsing Random Forest To Rank Feature Importance:\n")
    #Preparing Data
    x=df[feat_cols].fillna(df[feat_cols].mean()).values
    y=df['Cluster'].values

    #Standardizing
    std=StandardScaler()
    x_scaled=std.fit_transform(x)
    #Splitting Data
    x_train,x_test,y_train,y_test=ttsplit(
        x_scaled,y,test_size=0.3,random_state=42,stratify=y
    )
    print(f"\nTraining Random Forest Classifier...\n")
    print(f"Training Set: {len(x_train)} Cells")
    print(f"Test set: {len(x_test)} Cells")

    #Training Random Forest
    rf=RandomForestClassifier(n_estimators=200,max_depth=15,random_state=42,n_jobs=-1)
    rf.fit(x_train,y_train)
    #Evaluating
    train_acc=rf.score(x_train,y_train)
    test_acc=rf.score(x_test,y_test)
    print(f"\n Training Accuracy: {train_acc:.3f}")
    print(f"Test Accuracy: {test_acc:.3f}")
    #Getting Feature Importances
    importances=rf.feature_importances_
    #Creating DataFrame
    importance_df=pd.DataFrame({
        'Feature':[i.replace('Value of Default','')[:40] for i in feat_cols],
        'Importance':importances
    }).sort_values('Importance',ascending=False)
    print("\nTop 15 Most Important Features For Predicting Clusters\n")
    for idx,row in importance_df.head(15).iterrows():
        print(f"{row['Feature']:<42} {row['Importance']:.4f}")
    #Visualizing
    print("\nGenerating Feature Importance Plot...\n")
    top15=importance_df.head(15).sort_values('Importance',ascending=True)
    fig,ax=plt.subplots(figsize=(10,8))
    ax.barh(range(len(top15)),top15['Importance'],color='steelblue',alpha=0.8)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(top15['Feature'],fontsize=9)
    ax.set_xlabel('Feature Importance',fontsize=12)
    ax.set_title('Top 15 Features For Cluster Prediction (Random Forest)',fontsize=13)
    ax.grid(True,alpha=0.3,axis='x')
    plt.tight_layout()
    plt.show()
    return importance_df,rf

cluster_import,rf_cluster_mdl=feat_importance(
    df_fig5b_clustered,
    feat_cols,
    opt_k
)

In [ ]:
#Interpretation
def bio_interpretation(df,feat_cols,cluster_import_df=None):
    print("\nKey Finding:\n")
    #1.Shape Analysis
    if 'Shape_Category' in df.columns:
        shape_counts=df['Shape_Category'].value_counts()
        print("\n1.Shape Heterogeneity (Prolate/Oblate Analysis):\n")
        for category,count in shape_counts.items():
            if category!='Unknown':
                print(f"• {category}: {count} Cells ({count/len(df)*100:.1f}%)")
        print(f"\nInterpretation: Cells Show Diverse 3D Shape Distributions\n")
    #2.Clustering Results
    if 'Cluster' in df.columns:
        n_clusters=df['Cluster'].nunique()
        x_scaled=StandardScaler().fit_transform(df[feat_cols].fillna(df[feat_cols].mean()))
        sil_score=silhouette_score(x_scaled,df['Cluster'])
        print(f"\n2.Unsupervised Clustering:\n")
        print(f"• Optimal Clusters: {n_clusters}")
        print(f"• Silhouette Score: {sil_score:.3f}")
        if sil_score<0.3:
            print(f"• Interpretation: Weak Separation Suggests Continous Gradients")
            print(f"Rather Than Discrete Cell States")
        elif sil_score<0.5:
            print(f"• Interpretation: Moderate Separation-> Subtle But Detectable Populations")
        else:
            print(f"• Interpretation: Clear Distinct Populations")
        #Cluster Sizes
        cluster_counts=df['Cluster'].value_counts().sort_index()
        print(f"\nCluster Distribution:")
        for i,j in cluster_counts.items():
            print(f"Cluster {i}: {j} Cells ({j/len(df)*100:.1f}%)")

    #3.Validation
    if 'Shape_Category' in df.columns and 'Cluster' in df.columns:
        shape_numeric=pd.Categorical(df[df['Shape_Category']!='Unknown']['Shape_Category']).codes
        cluster_numeric=df[df['Shape_Category']!='Unknown']['Cluster'].values
        ari_score=adjusted_rand_score(shape_numeric,cluster_numeric)
        print(f"\n3.Cluster Validation:")
        print(f"• Adjusted Rand Index (Clusters Vs. Shapes): {ari_score:.3f}")
        if ari_score>0.3:
            print(f"• Interpretation: Clusters Partially Align With Shape Categories")
        elif ari_score>0.1:
            print(f"• Interpretation: Weak Agreement-> Clusters Capture Additional Patterns")
        else:
            print(f"• Interpretation: Clusters Independent Of Shape-> Driven By Other Features")

    #4.Feature Importance
    if cluster_import_df is not None:
        print(f"\n4.Key Discriminative Features:\n")
        print(f"Top 5 Features Driving Cluster Differences:")
        for idx,row in cluster_import_df.head(5).iterrows():
            print(f"• {row['Feature']}")

bio_interpretation(
    df_fig5b_clustered,
    feat_cols,
    cluster_import_df=cluster_import
)
